In [1]:
!nvcc --version

/bin/bash: line 1: nvcc: command not found


In [1]:
!wget https://raw.githubusercontent.com/yoosufaamir/HPC-/main/passwords.txt -O passwords.txt

--2026-10-08 17:56:08--  https://raw.githubusercontent.com/yoosufaamir/HPC-/main/passwords.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 110000 (107K) [text/plain]
Saving to: ‘passwords.txt’

passwords.txt       100%[===================>] 107.42K  --.-KB/s    in 0.01s   

2026-10-08 17:56:08 (9.04 MB/s) - ‘passwords.txt’ saved [110000/110000]



In [15]:
%%writefile crack.cu
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

#define RAW_LEN 4          /* aa99 -> 4 characters                 */
#define ENC_LEN 10         /* encrypted password length (no \0)    */
#define LINE_BUF 32


__device__ void cudaCrypt(const char *rawPassword, char *newPassword) {

    newPassword[0] = rawPassword[0] + 2;
    newPassword[1] = rawPassword[0] - 2;
    newPassword[2] = rawPassword[0] + 1;
    newPassword[3] = rawPassword[1] + 3;
    newPassword[4] = rawPassword[1] - 3;
    newPassword[5] = rawPassword[1] - 1;
    newPassword[6] = rawPassword[2] + 2;
    newPassword[7] = rawPassword[2] - 2;
    newPassword[8] = rawPassword[3] + 4;
    newPassword[9] = rawPassword[3] - 4;

    for (int i = 0; i < 10; i++) {
        if (i >= 0 && i < 6) {               /* lowercase-letter section */
            if (newPassword[i] > 122) {
                newPassword[i] = (newPassword[i] - 122) + 97;
            } else if (newPassword[i] < 97) {
                newPassword[i] = (97 - newPassword[i]) + 97;
            }
        } else {                             /* digit section */
            if (newPassword[i] > 57) {
                newPassword[i] = (newPassword[i] - 57) + 48;
            } else if (newPassword[i] < 48) {
                newPassword[i] = (48 - newPassword[i]) + 48;
            }
        }
    }
}


__device__ int devStrEqual(const char *a, const char *b, int len) {
    for (int i = 0; i < len; i++) {
        if (a[i] != b[i]) return 0;
    }
    return 1;
}


__global__ void crackKernel(const char *d_encrypted, char *d_decrypted, int n) {

    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx >= n) {
        return;   /* guard - last block may have fewer passwords than threads */
    }

    const char *target = d_encrypted + (idx * ENC_LEN);
    char *outSlot = d_decrypted + (idx * (RAW_LEN + 1));

    char guess[RAW_LEN];
    char encryptedGuess[ENC_LEN];

    for (int a = 0; a < 26; a++) {                 /* letter 1 */
        guess[0] = (char)('a' + a);
        for (int b = 0; b < 26; b++) {              /* letter 2 */
            guess[1] = (char)('a' + b);
            for (int c = 0; c < 10; c++) {          /* digit 1 */
                guess[2] = (char)('0' + c);
                for (int d = 0; d < 10; d++) {      /* digit 2 */
                    guess[3] = (char)('0' + d);

                    cudaCrypt(guess, encryptedGuess);

                    if (devStrEqual(encryptedGuess, target, ENC_LEN)) {
                        outSlot[0] = guess[0];
                        outSlot[1] = guess[1];
                        outSlot[2] = guess[2];
                        outSlot[3] = guess[3];
                        outSlot[4] = '\0';
                        return;   /* found it - stop searching */
                    }
                }
            }
        }
    }

    /* no match found (shouldn't happen with well-formed input) */
    outSlot[0] = '?'; outSlot[1] = '?'; outSlot[2] = '?'; outSlot[3] = '?';
    outSlot[4] = '\0';
}


#define CUDA_CHECK(call)                                                    \
    do {                                                                    \
        cudaError_t err = (call);                                           \
        if (err != cudaSuccess) {                                           \
            fprintf(stderr, "CUDA error at %s:%d - %s\n",                   \
                    __FILE__, __LINE__, cudaGetErrorString(err));           \
            exit(EXIT_FAILURE);                                             \
        }                                                                   \
    } while (0)

int main(int argc, char *argv[]) {

    if (argc != 2) {
        printf("Usage: %s <encryptedPasswordsFile>\n", argv[0]);
        return 1;
    }



    FILE *fp = fopen(argv[1], "r");
    if (fp == NULL) {
        perror("Error opening input file");
        return 1;
    }

    int capacity = 1024;
    int n = 0;
    char *h_encrypted = (char *)malloc(capacity * ENC_LEN);

    char line[LINE_BUF];
    while (fgets(line, LINE_BUF, fp) != NULL) {


        line[strcspn(line, "\r\n")] = '\0';

        if (strlen(line) != ENC_LEN) {
            continue;   /* skip malformed / blank lines */
        }

        if (n >= capacity) {
            capacity *= 2;
            h_encrypted = (char *)realloc(h_encrypted, capacity * ENC_LEN);
        }

        memcpy(h_encrypted + (n * ENC_LEN), line, ENC_LEN);
        n++;
    }
    fclose(fp);

    if (n == 0) {
        printf("No valid %d-character encrypted passwords found in file.\n", ENC_LEN);
        free(h_encrypted);
        return 1;
    }

    printf("Read %d encrypted passwords from %s\n", n, argv[1]);



    char *d_encrypted = NULL;
    char *d_decrypted = NULL;

    size_t encBytes = (size_t)n * ENC_LEN;
    size_t decBytes = (size_t)n * (RAW_LEN + 1);

    CUDA_CHECK(cudaMalloc((void **)&d_encrypted, encBytes));
    CUDA_CHECK(cudaMalloc((void **)&d_decrypted, decBytes));

    CUDA_CHECK(cudaMemcpy(d_encrypted, h_encrypted, encBytes, cudaMemcpyHostToDevice));



    int threadsPerBlock = (n < 1000) ? n : 1000;
    int blocks = (n + threadsPerBlock - 1) / threadsPerBlock;

    printf("Launching kernel with %d blocks x %d threads (%d threads total for %d passwords)\n",
           blocks, threadsPerBlock, blocks * threadsPerBlock, n);

    crackKernel<<<blocks, threadsPerBlock>>>(d_encrypted, d_decrypted, n);

    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());


    char *h_decrypted = (char *)malloc(decBytes);
    CUDA_CHECK(cudaMemcpy(h_decrypted, d_decrypted, decBytes, cudaMemcpyDeviceToHost));


    FILE *out = fopen("decrypted.txt", "w");
    if (out == NULL) {
        perror("Error opening output file");
    } else {
        for (int i = 0; i < n; i++) {
            const char *enc = h_encrypted + (i * ENC_LEN);
            const char *dec = h_decrypted + (i * (RAW_LEN + 1));
            fprintf(out, "%.*s -> %s\n", ENC_LEN, enc, dec);
        }
        fclose(out);
        printf("Decrypted passwords written to decrypted.txt\n");
    }


    CUDA_CHECK(cudaFree(d_encrypted));
    CUDA_CHECK(cudaFree(d_decrypted));

    free(h_encrypted);
    free(h_decrypted);

    return 0;
}

Overwriting crack.cu


In [16]:
!nvcc crack.cu -o PWCrack

In [17]:
!./PWCrack passwords.txt

Read 10000 encrypted passwords from passwords.txt
Launching kernel with 10 blocks x 1000 threads (10000 threads total for 10000 passwords)
Decrypted passwords written to decrypted.txt
